In [ ]:
import xarray as xr
import uxarray as ux

# Simple if check

Minimal code, easy to understand/implement, but we anticipate having to do this all over the place in the future, and we want to separate the backend logic divergence from the science routines

In [ ]:
def zonal_mean(da, lat=None):
    if hasattr(da, "zonal_mean"):
        return da.zonal_mean(lat=lat)

    out = da.mean(dim="lon")

    if lat is not None:
        ...
    return out

# Adaptor Backend  (recommended)

Separates concerns of science logic and backend functional divergence, scalable for future divergences between UXarray and Xarray, saves metadata, but is a little heavier to implement

In [ ]:
class BaseBackend:
    def zonal_mean(self, da, lat=None):
        raise NotImplementedError


class UXarrayBackend(BaseBackend):
    def zonal_mean(self, da, lat=None):
        return da.zonal_mean(lat=lat)


class XarrayBackend(BaseBackend):

    def __init__(self, ds):

        self.lon_name = _find_var(
            ds,
            standard_name='longitude',
            long_name='longitude',
            possible_names=[
                'lon',
                'longitude',
                'x',
                'rlon',
            ],
            description='longitude dimension',
        )

        self.lat_name = _find_var(
            ds,
            standard_name='latitude',
            long_name='latitude',
            possible_names=[
                'lat',
                'latitude',
                'y',
                'rlat',
            ],
            description='latitude coordinate',
        )

    def zonal_mean(self, da, lat=None):

        if self.lon_name not in da.dims:
            raise ValueError(
                f"Longitude dimension '{self.lon_name}' "
                f"not found in {da.dims}"
            )

        out = da.mean(dim=self.lon_name)

        if lat is not None:
            out = self._subset_lat(out, lat)

        return out

    def _subset_lat(self, da, lat):
        if isinstance(lat, tuple):
            start, stop, step = lat
            target_lats = np.arange(start, stop + step, step)
            return da.sel(lat=target_lats, method="nearest")

        elif np.isscalar(lat):
            return da.sel(lat=lat, method="nearest")

        else:
            return da.sel(lat=lat, method="nearest")


def get_backend(ds):
    try:
        import uxarray as ux

        if isinstance(ds, ux.UxDataset):
            return UXarrayBackend()
    except ImportError:
        pass

    return XarrayBackend()

In [ ]:
# Then inside zonal_mpsi

backend = get_backend(ds)

da_v_zonal = backend.zonal_mean(ux_ipress, lat=lat)
da_PS_zonal = backend.zonal_mean(
    ds[surface_air_pressure_varname],
    lat=lat,
)

# Single Dispatch

Python native way to handle different versions of the same function based on what the input args are

Short term solution, only helps this one function instead of function operations that are added

Not great for cacheing our coordinate names, meant for "stateless"

In [ ]:
from functools import singledispatch

@singledispatch
def zonal_mean(obj, lat=None):
    raise TypeError(f"Unsupported type: {type(obj)}")


@zonal_mean.register
def _(obj: ux.UxDataArray, lat=None):
    return obj.zonal_mean(lat=lat)


@zonal_mean.register
def _(obj: xr.DataArray, lat=None):
    out = obj.mean(dim="lon")

    if lat is not None:
        ...
    return out


da_v_zonal = zonal_mean(ux_ipress, lat=lat)

# Accessor API

Xarray-native

Intended for public user Xarray functions w/o monkeypatching, not as helpful as it is only being used "under the hood"/backend/helper functions

In [ ]:
@xr.register_dataarray_accessor("xra")
class XRAccessor:
    def __init__(self, xarray_obj):
        self._obj = xarray_obj

    def zonal_mean(self, lat=None, lon_name="lon"):
        out = self._obj.mean(dim=lon_name)

        ...
        return out


da.xra.zonal_mean(lat=lat)